# 02 — AI Agents (Task 2)

**Goals (Milestone 2):**

1. Wire AIMA's `alpha_beta_cutoff_search` to our 6×6 game.
2. Define the **heuristic evaluation function** (`tictactoe66.Heuristic`).
3. Build configurable players via `make_alpha_beta_player(depth, weights)`.
4. Demonstrate two AI agents playing a complete game.
5. Quick sanity tournament: alpha-beta vs random baseline.

In [11]:
import os, sys, random, time
_here = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _here not in sys.path:
    sys.path.insert(0, _here)
import settings, tictactoe66
from games4e import GameState

## 1. The heuristic

`Heuristic` walks every length-`k` window on the board and:

* **adds** `w_two` / `w_three` for own-only windows containing 2 / 3 markers,
* **subtracts** `w_block_two` / `w_block_three` for opponent-only windows of equal danger,
* **adds** `w_center` for own markers in the central 2×2,
* short-circuits to `±w_win` for terminal states.

The score is computed from the *current to-move player's* perspective, which is what `alpha_beta_cutoff_search` expects.

In [12]:
game = tictactoe66.TicTacToe66()
h = tictactoe66.Heuristic(game=game, perspective='X')

# Demo state: X has 3-in-row; O has nothing comparable
demo_board = {(2,2):'X',(2,3):'X',(2,4):'X',(5,5):'O'}
demo_state = GameState(to_move='O', utility=0, board=demo_board,
                       moves=[(x,y) for x in range(1,7) for y in range(1,7) if (x,y) not in demo_board])
game.display(demo_state)
print('heuristic for X:', h(demo_state))
print('heuristic for O:', tictactoe66.Heuristic(game=game, perspective='O')(demo_state))

     1  2  3  4  5  6
   +------------------+
 1 |  .  .  .  .  .  . |
 2 |  .  X  X  X  .  . |
 3 |  .  .  .  .  .  . |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  .  O  . |
 6 |  .  .  .  .  .  . |
   +------------------+
heuristic for X: 21.0
heuristic for O: -17.0


## 2. Alpha-beta cutoff player

`make_alpha_beta_player(depth)` returns a function with the AIMA player signature `(game, state) -> move` so we can pass it straight to `play_one_game`.

In [13]:
ab_d2 = tictactoe66.make_alpha_beta_player(depth=2, name='AB-d2')
ab_d3 = tictactoe66.make_alpha_beta_player(depth=3, name='AB-d3')

rng = random.Random(settings.RANDOM_SEED)
t0 = time.perf_counter()
res = tictactoe66.play_one_game(game, ab_d2, ab_d2, n_prealloc=2, rng=rng, display=False)
elapsed = time.perf_counter() - t0
print(res, f'elapsed={elapsed:.2f}s')

{'winner': 'X', 'plies': 9, 'n_prealloc': 2} elapsed=0.20s


## 3. AI vs AI demo with display

In [14]:
rng = random.Random(123)
res = tictactoe66.play_one_game(game, ab_d2, ab_d2, n_prealloc=1, rng=rng, display=True)
print('outcome:', res)

     1  2  3  4  5  6
   +------------------+
 1 |  .  .  .  X  .  . |
 2 |  .  .  .  .  .  . |
 3 |  .  .  .  .  .  O |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+

AB-d2 -> (1, 3)
     1  2  3  4  5  6
   +------------------+
 1 |  .  .  X  X  .  . |
 2 |  .  .  .  .  .  . |
 3 |  .  .  .  .  .  O |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+

AB-d2 -> (4, 6)
     1  2  3  4  5  6
   +------------------+
 1 |  .  .  X  X  .  . |
 2 |  .  .  .  .  .  . |
 3 |  .  .  .  .  .  O |
 4 |  .  .  .  .  .  O |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+

AB-d2 -> (1, 2)
     1  2  3  4  5  6
   +------------------+
 1 |  .  X  X  X  .  . |
 2 |  .  .  .  .  .  . |
 3 |  .  .  .  .  .  O |
 4 |  .  .  .  .  .  O |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+

AB-d2 -> (1, 1)
     1  2  3  4  5  6
   +------------------+
 1 |  

## 4. Sanity tournament: alpha-beta vs random

A small N is enough here — notebook 03 runs the rigorous sweep.

In [15]:
import collections
N = 10
rng = random.Random(0)
results = collections.Counter()
for i in range(N):
    r = tictactoe66.play_one_game(game, ab_d2, tictactoe66.random_legal_player,
                              n_prealloc=2, rng=rng)
    results[r['winner']] += 1
print('AB-d2 (X) vs Random (O) over', N, 'games:', dict(results))

AB-d2 (X) vs Random (O) over 10 games: {'X': 10}


**Milestone 2 reached.** AI agents play full games correctly. Notebook 03 sweeps depth and heuristic weights to find a stronger configuration.